# E-posta Spam Sınıflandırma

Bu notebook'ta etiketli e-postaları **normal (ham)** ve **spam** olarak sınıflandırmak için dört makine öğrenmesi algoritmasını karşılaştıracağız.

**Önemli:** UCI Spambase, ham e-posta metnini değil, e-postalardan önceden çıkarılmış sayısal özellikleri içerir. Burada metin temizleme yerine sınıflandırma ve model değerlendirmeye odaklanıyoruz.

Veri kaynağı: [UCI Spambase](https://archive.ics.uci.edu/dataset/94/spambase).

## 1. Gerekli kütüphaneyi kur

Colab'da bu hücre UCI veri setini indirmek için gereken paketi kurar. Hücreyi çalıştırmak için solundaki oynat düğmesine bas.

In [ ]:
%pip -q install ucimlrepo

## 2. Araçları içe aktar

- pandas: tablo biçimindeki veriyi incelemek için.
- scikit-learn: veriyi bölmek, modelleri eğitmek ve ölçmek için.
- matplotlib: confusion matrix grafiklerini çizmek için.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42

## 3. Veri setini yükle ve tanı

UCI sayfasındaki Spambase veri setini indiriyoruz. Hedef sütundaki 0 normal e-postayı, 1 spam e-postayı gösteriyor. Önce örnek ve özellik sayısını, ardından sınıf dağılımını kontrol ediyoruz.

In [ ]:
spambase = fetch_ucirepo(id=94)
X = spambase.data.features.copy()
y = spambase.data.targets.iloc[:, 0].astype(int).copy()

print(f"Örnek sayısı: {len(X)}")
print(f"Özellik sayısı: {X.shape[1]}")
print("\nSınıf dağılımı (0 = normal, 1 = spam):")
display(y.value_counts().sort_index().rename(index={0: "Normal", 1: "Spam"}).to_frame("E-posta sayısı"))
display(X.head())

## 4. Eğitim ve test verisini ayır

Modelin başarısını daha önce görmediği e-postalarda ölçmek için veriyi iki parçaya ayırıyoruz. Yüzde 80 eğitimde, yüzde 20 testte kalır. stratify=y iki parçadaki normal/spam oranlarının benzer kalmasına yardım eder.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Eğitim örneği: {len(X_train)}")
print(f"Test örneği: {len(X_test)}")

## 5. Dört algoritmayı tanımla

Bunlar aynı işi farklı yöntemlerle öğrenir. Lojistik Regresyon ve SVM öncesinde özellikleri ölçekliyoruz; Random Forest ve Naive Bayes için bu adım gerekmiyor.

In [ ]:
models = {
    "Gaussian Naive Bayes": GaussianNB(),
    "Lojistik Regresyon": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    ),
    "Destek Vektör Makinesi (RBF)": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf"),
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

## 6. Modelleri eğit ve test et

Her model yalnızca eğitim bölümünde öğrenir. Ardından test bölümündeki tahminleri accuracy, precision, recall ve F1 ile ölçeriz. Spam sınıfı 1 olduğu için precision, recall ve F1 hesaplarında pozitif sınıf olarak 1 kullanıyoruz.

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (spam)": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall (spam)": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1 (spam)": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
    })

results_df = pd.DataFrame(results).set_index("Model").sort_values("F1 (spam)", ascending=False)
display(results_df.style.format("{:.3f}"))

### Ölçütleri nasıl okuyacağız?

- **Accuracy:** Tüm e-postaların ne kadarını doğru sınıflandırdı?
- **Precision (spam):** Spam dediği e-postaların ne kadarı gerçekten spam?
- **Recall (spam):** Gerçek spam e-postaların ne kadarını yakaladı?
- **F1 (spam):** Precision ve recall'u birlikte özetler.

Bu notebook'ta en yüksek F1 değerini karşılaştırma için kullanıyoruz. Yine de yanlış spam alarmı (FP) ile kaçırılan spam (FN) sayılarına da bakmak gerekir.

## 7. Confusion matrix'leri karşılaştır

Her grafikte **satırlar gerçek sınıfı**, **sütunlar modelin tahminini** gösterir. Sınıf sırası normal, sonra spam'dir:

[[TN, FP], [FN, TP]]

Normal e-postayı spam sanmak **FP**, spam'i normal sanmak **FN** olur.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, (name, y_pred) in zip(axes.flat, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        labels=[0, 1],
        display_labels=["Normal", "Spam"],
        values_format="d",
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()

## 8. En yüksek F1 alan modeli incele

Aşağıdaki model adı ve rapor, yukarıdaki gerçek sonuçlardan otomatik çıkar. Rapor her sınıf için precision, recall ve F1 değerlerini gösterir.

In [ ]:
best_model_name = results_df.index[0]
best_prediction = predictions[best_model_name]

print(f"En yüksek F1 alan model: {best_model_name}")
print("\nSınıflandırma raporu:")
print(classification_report(
    y_test,
    best_prediction,
    labels=[0, 1],
    target_names=["Normal", "Spam"],
    zero_division=0,
))

## 9. Rapor için sonuçları not et

1. Sonuç tablosundaki accuracy, precision, recall ve F1 değerlerini report_template.md dosyasına aktar.
2. En yüksek F1 alan modeli seç; seçimini precision/recall dengesiyle açıkla.
3. Seçilen modelin confusion matrix'inden FP ve FN sayılarını yaz.
4. UCI Spambase'in ham e-posta metni değil sayısal özellikler içerdiğini sınırlama olarak belirt.

**Not:** Bu notebook her çalıştırıldığında veri bölümü aynı kalır. Farklı veri setlerinde sonuçlar değişebilir.